In [1]:
import pandas as pd
from geopy.geocoders import Nominatim
import time

In [ ]:
df = pd.read_excel('Name of File.csv or .xlsx')
print(df.head(5))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

# Convert coordinates to radians
coords = np.radians(df[['Latitude', 'Longitude']].values)

# Build BallTree using Haversine distance
tree = BallTree(coords, metric='haversine')

# Define search radius (convert km to radians)
radius_km = 1
radius = radius_km / 6371.0  # Earth radius in km

# Find neighbours within radius for each point
indices = tree.query_radius(coords, r=radius)

# Store neighbours (excluding self)
df['neighbours'] = [list(neigh[neigh != i]) for i, neigh in enumerate(indices)]

# Add corresponding PU names for neighbours
df['neighbours_pu_name'] = [
    df.loc[neigh, 'PU-Name'].tolist() for neigh in df['neighbours']
]

print("Neighbour search complete!")


Neighbour search complete!


In [ ]:
print(df.head(10).to_string(index=False))

In [5]:
# Suppose these are your party columns
parties = ['APC', 'PDP', 'LP', 'NNPP',]

# Initialize columns for outlier scores
for party in parties:
    df[f'{party}_outlier_score'] = np.nan

# Function to compute outlier scores
for i, row in df.iterrows():
    neighbours = df.loc[row['neighbours']]
    for party in parties:
        if len(neighbours) > 0:
            neighbour_mean = neighbours[party].mean()
            neighbour_std = neighbours[party].std()
            if neighbour_std > 0:
                score = abs(row[party] - neighbour_mean) / neighbour_std
                df.loc[i, f'{party}_outlier_score'] = score


In [ ]:
print(df.head(10).to_string())

In [9]:
df['neighbours'] = df['neighbours'].apply(lambda x: ', '.join(str(int(i)) for i in x))
df['neighbours_pu_name'] = df['neighbours_pu_name'].apply(lambda x: ', '.join(x))
df.to_csv('cleaned_neighbours.csv', index=False)


In [8]:
# Didnt later use this but keeping for reference
df.to_excel('excel_final_with_outlier_lat_long.xlsx')


In [ ]:
for party in parties:
    top_outliers = df.sort_values(by=f'{party}_outlier_score', ascending=False).head(10)
    print(f"Top outliers for {party}:")
    print(top_outliers[['PU-Name', f'{party}_outlier_score', 'neighbours', 'neighbours_pu_name']].to_string())
    print("\n" + "-"*80 + "\n")


In [78]:
import pandas as pd

# Define your Excel file name
output_file = "all_outliers_per_party.xlsx"

# Create an Excel writer
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for party in parties:
        # Sort all rows for the party
        party_outliers = df.sort_values(by=f'{party}_outlier_score', ascending=False).copy()
        party_outliers['party'] = party

        # Add the original index as a column
        party_outliers.reset_index(inplace=True)
        party_outliers.rename(columns={'index': 'PU_Index'}, inplace=True)

        # Convert lists to readable comma-separated strings
        party_outliers['neighbours'] = party_outliers['neighbours'].apply(
            lambda x: ', '.join(str(int(i)) for i in x) if isinstance(x, (list, tuple)) else str(x)
        )
        party_outliers['neighbours_pu_name'] = party_outliers['neighbours_pu_name'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, (list, tuple)) else str(x)
        )

        # Select relevant columns
        export_cols = ['PU_Index', 'party', 'PU-Name', f'{party}_outlier_score', 'neighbours', 'neighbours_pu_name']

        # Write to new sheet
        party_outliers[export_cols].to_excel(writer, sheet_name=party, index=False)

print("✅ Export complete! Each sheet now includes the PU_Index column showing the original DataFrame index.")


✅ Export complete! Each sheet now includes the PU_Index column showing the original DataFrame index.


In [90]:
# Save to CSV
top3_all.to_csv('top3_overall_outliers.csv', index=False)

# Save styled table to HTML (for viewing in browser)
styled_table.to_html('top3_overall_outliers.html')
